In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

In [2]:
def get_data(url):
    html_page = requests.get(url)
    soup = BeautifulSoup(html_page.content, 'html.parser')
    
    def gnp(soup):
        next_button = soup.find("li",
                                class_="next")  #May return none if on final page
        if next_button:
            return next_button.find('a').attrs['href']
        else:
            return None
    
    def url_list(url,lll=[]):
        lll.append(url)
        next_url_ext = gnp(soup)
        if next_url_ext:
            next_url = '/'.join(url.split('/')[:-1]) + '/' + next_url_ext
            return url_list(next_url)
        else:
            return lll
    
    def retrieve_titles(soup):
        warning = soup.find('div', class_="alert alert-warning")
        book_container = warning.nextSibling.nextSibling
        titles = [
            h3.find('a').attrs['title'] for h3 in book_container.findAll('h3')
        ]
        return titles

    def retrieve_ratings(soup):
        import re
        warning = soup.find('div', class_="alert alert-warning")
        book_container = warning.nextSibling.nextSibling
        regex = re.compile("star-rating (.*)")
        star_ratings = []
        for p in book_container.findAll('p', {"class": regex}):
            star_ratings.append(p.attrs['class'][-1])
        star_dict = {
            'One': 1,
            'Two': 2,
            'Three': 3,
            'Four': 4,
            'Five': 5
        }  # Manually create a dictionary to translate to numeric
        star_ratings = [star_dict[s] for s in star_ratings]
        return star_ratings

    def retrieve_prices(soup):
        warning = soup.find('div', class_="alert alert-warning")
        book_container = warning.nextSibling.nextSibling
        prices = [
            float(p.text[1:])
            for p in book_container.findAll('p', class_="price_color")
        ]  # Removing the pound sign and converting to float
        return prices

    def retrieve_availabilities(soup):
        warning = soup.find('div', class_="alert alert-warning")
        book_container = warning.nextSibling.nextSibling
        avails = [
            a.text.strip()
            for a in book_container.findAll('p', class_="instock availability")
        ]
        return avails
    
    links_list = url_list(url)
    df = pd.DataFrame()
    for link in links_list:
        url = link
#         html_page = requests.get(url)
#         soup = BeautifulSoup(html_page.content, 'html.parser')
        warning = soup.find('div', class_="alert alert-warning")
        book_container = warning.nextSibling.nextSibling
        df_1 = pd.DataFrame([
            retrieve_titles(soup),
            retrieve_ratings(soup),
            retrieve_prices(soup),
            retrieve_availabilities(soup)
        ]).transpose()
        df_1.columns = ['Title', 'Star_Rating', 'Price_(pounds)', 'Availability']
        df = df.append(df_1)
        df_c = df.reset_index().drop(columns=['index'])
    return df_c

In [ ]:
get_data('http://books.toscrape.com/')